<a href="https://colab.research.google.com/github/ArkanUbaidillah/BigData26_B_2411537001_ArkanUbaidillahWarman/blob/main/Praktikum2/Lat_BD_P02_2411537001_ArkanUbaidillahWarman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install faker -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 13.1 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from faker import Faker
import random

In [4]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + "  "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])  # rating opsional

    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


In [5]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


In [6]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 15
transaction_id duplicate: 15
Jumlah baris setelah drop_duplicates(): 500


In [7]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

In [8]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

In [9]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

In [10]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

In [11]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

In [12]:
print("Jumlah nilai unik category:", df["category"].nunique())
print("Jumlah nilai unik payment_method:", df["payment_method"].nunique())
print(df[["price", "quantity"]].dtypes)
print("Contoh transaction_date:", df["transaction_date"].head(3).tolist())

Jumlah nilai unik category: 6
Jumlah nilai unik payment_method: 4
price       float64
quantity      int64
dtype: object
Contoh transaction_date: ['2026-07-15', '2026-07-11', '2026-08-27']


## Q. Latihan

### Latihan 1

Ubah SEED menjadi 7 dan jalankan ulang seluruh pipeline. Bandingkan jumlah baris transaksi_mentah.csv dan transaksi_bersih.csv dengan hasil SEED = 42. Apakah jumlahnya sama? Jelaskan mengapa.

Agar perbandingan dapat dijalankan dalam satu notebook tanpa mengubah sel utama, langkah K-2 sampai K-6 dibungkus menjadi fungsi `jalankan_pipeline(seed)` dengan isi yang identik, lalu fungsi itu dipanggil untuk SEED = 7 dan SEED = 42.

In [13]:
def jalankan_pipeline(seed):
    np.random.seed(seed)
    random.seed(seed)
    fake_lokal = Faker("id_ID")
    Faker.seed(seed)

    rows = []
    for i in range(1, N + 1):
        trx_id = f"TRX{i:05d}"
        nama_pelanggan = fake_lokal.name()
        produk = fake_lokal.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
        kategori = random.choice(kategori_produk)
        harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
        qty = random.randint(1, 5)
        harga_variants = [
            str(harga_dasar),
            f"Rp{harga_dasar:,}".replace(",", "."),
            f"{harga_dasar}.0",
            f" {harga_dasar} ",
        ]
        harga = random.choice(harga_variants)
        tgl = fake_lokal.date_between(start_date="-90d", end_date="today")
        tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
        tanggal = random.choice(tgl_variants)
        metode = random.choice(metode_bayar)
        if random.random() < 0.3:
            metode = metode.lower()
        if random.random() < 0.2:
            kategori = kategori.upper() + "  "
        kota = fake_lokal.city()
        rating = random.choice([1, 2, 3, 4, 5, None, None])
        rows.append({
            "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
            "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
            "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
        })

    df_lokal = pd.DataFrame(rows)
    for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
        idx = df_lokal.sample(frac=frac, random_state=seed).index
        df_lokal.loc[idx, col] = np.nan
    dup_rows = df_lokal.sample(n=15, random_state=seed)
    df_lokal = pd.concat([df_lokal, dup_rows], ignore_index=True)
    df_lokal = df_lokal.sample(frac=1, random_state=seed).reset_index(drop=True)
    n_mentah = len(df_lokal)

    df_lokal = df_lokal.drop_duplicates()
    df_lokal = df_lokal.dropna(subset=["customer_name", "payment_method"])
    df_lokal["shipping_city"] = df_lokal["shipping_city"].fillna("Tidak Diketahui")
    for col in ["category", "payment_method", "shipping_city"]:
        df_lokal[col] = df_lokal[col].astype("string").str.strip().str.title()
    df_lokal["payment_method"] = df_lokal["payment_method"].replace({"Cod": "COD"})
    df_lokal["price"] = df_lokal["price"].apply(bersihkan_harga)
    df_lokal["transaction_date"] = df_lokal["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")
    df_lokal["quantity"] = df_lokal["quantity"].astype(int)
    df_lokal["price"] = df_lokal["price"].astype(float)
    n_bersih = len(df_lokal)
    return n_mentah, n_bersih

mentah_7, bersih_7 = jalankan_pipeline(7)
mentah_42, bersih_42 = jalankan_pipeline(42)
print("SEED = 7  : transaksi_mentah", mentah_7, "baris, transaksi_bersih", bersih_7, "baris")
print("SEED = 42 : transaksi_mentah", mentah_42, "baris, transaksi_bersih", bersih_42, "baris")

SEED = 7  : transaksi_mentah 515 baris, transaksi_bersih 490 baris
SEED = 42 : transaksi_mentah 515 baris, transaksi_bersih 490 baris


**Jawaban Latihan 1.** Jumlah barisnya sama: 515 baris mentah dan 490 baris bersih untuk kedua seed. Alasannya, jumlah baris tidak ditentukan oleh nilai seed melainkan oleh parameter yang tetap: N = 500 transaksi ditambah 15 baris duplikat menghasilkan 515 baris mentah, dan jumlah baris yang dibuang ditentukan oleh fraksi missing value (2 persen customer_name, 1,5 persen payment_method) serta 15 baris duplicate yang semuanya bernilai tetap. Seed hanya mengubah baris mana yang terpilih dan nilai acak seperti nama, kota, dan rating, bukan berapa banyak baris yang terpilih. Karena itu pipeline dengan SEED = 7 tetap berakhir pada 490 baris bersih, hanya isi barisnya yang berbeda.

### Latihan 2

Tambahkan kolom is_valid_price bernilai True jika price > 0. Gunakan untuk memeriksa apakah ada harga tidak valid.

In [14]:
df["is_valid_price"] = df["price"] > 0
print("Jumlah baris dengan harga tidak valid:", (~df["is_valid_price"]).sum())
print(df["is_valid_price"].value_counts())

Jumlah baris dengan harga tidak valid: 0
is_valid_price
True    490
Name: count, dtype: int64


Tidak ada harga tidak valid pada dataset bersih: seluruh 490 baris bernilai True karena fungsi `bersihkan_harga` sudah mengubah semua variasi teks harga menjadi angka positif, dan baris yang gagal dikonversi akan menjadi NaN sehingga ikut terdeteksi oleh pemeriksaan ini.

### Latihan 3

Hitung jumlah transaksi per category menggunakan value_counts() pada dataset yang sudah bersih.

In [15]:
print(df["category"].value_counts())

category
Olahraga        97
Kesehatan       91
Elektronik      89
Buku            82
Fashion         66
Rumah Tangga    65
Name: count, dtype: Int64
